<a href="https://colab.research.google.com/github/Awaterr/myweibo/blob/main/_downloads/070179efc13bd796c5dd4af7bf52d5b9/intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

In [3]:
import torch
import numpy as np

**初始化张量**


直接来自数据

In [4]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

来自 NumPy 数组

In [5]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

来自另一个张量：

In [6]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.8012, 0.7603],
        [0.4882, 0.0567]]) 



取随机值或固定值：

In [7]:
shape = (2,3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.1643, 0.8336, 0.4694],
        [0.8809, 0.0109, 0.2636]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


**张量的属性**

In [8]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


**张量运算**:
显式地将张量移动到加速器

In [9]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

In [10]:
import torch

# 创建一个张量（默认在CPU）
tensor = torch.rand((2,3))
print("移动前设备：", tensor.device)

# 教程里的代码
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())

print("移动后设备：", tensor.device)


移动前设备： cpu
移动后设备： cpu


张量不能跨设备运算

In [11]:
# gpu张量
tensor_gpu = torch.rand(2,3).to("cuda")
# cpu张量
tensor_cpu = torch.rand(2,3)

# 直接相加，会报错！
result = tensor_gpu + tensor_cpu
print(result)


AssertionError: Torch not compiled with CUDA enabled

标准的类NumPy索引和切片：

In [12]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


连接张量:

In [13]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])


**算术运算**

一、矩阵乘法（矩阵点乘，不是挨个数字相乘）:

In [14]:
# 三种等价写法，y1=y2=y3，结果完全一样
# a @ b / matmul
y1 = tensor @ tensor.T          # @ 符号，PyTorch矩阵乘法简写
y2 = tensor.matmul(tensor.T)    # .matmul() 实例方法

y3 = torch.rand_like(y1)        # 创建一个和y1形状一样、随机值的张量，用来存结果
torch.matmul(tensor, tensor.T, out=y3) # 函数写法，out=y3：直接把计算结果放进y3，不新建内存


tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])

二、element-wise product 逐元素相乘（哈达玛积，符号 `*`）：

In [15]:
# 三种等价写法，z1=z2=z3，结果完全一样
#a * b / mul
z1 = tensor * tensor            # * 星号，逐元素相乘简写
z2 = tensor.mul(tensor)         # .mul() 实例方法

z3 = torch.rand_like(tensor)    # 创建同shape随机张量
torch.mul(tensor, tensor, out=z3) # 函数写法，out=z3：结果直接写入z3，节省内存


tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

单元素张量转换为Python数值item()：

In [16]:
agg = tensor.sum()#对张量所有元素求和，返回的仍然是一个0 维张量（标量张量），类型是 torch.Tensor。
agg_item = agg.item()#.item() 方法：把只有单个数值的标量张量，转成 Python 原生数字（int/float）
print(agg_item, type(agg_item))

12.0 <class 'float'>


**原地操作** 将结果存储到操作数中的操作称为原地操作。

原地操作可以节省一些内存，但由于会立即丢失历史数据，因此在计算导数时可能会出现问题。因此，不建议使用原地操作。

In [17]:
print(f"{tensor} \n")
tensor.add_(5)   # add_ 末尾有下划线！原地加5
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


# **与NumPy桥接**

CPU上的张量和NumPy数组可以共享底层内存位置，更改其中一个也会更改另一个。

**张量转NumPy数组**

In [19]:
t = torch.ones(5)    # 创建一个长度为5、全部是1的PyTorch张量
print(f"t: {t}")
n = t.numpy()         # .numpy()：把torch张量转成numpy数组
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]


张量的变化会反映在 NumPy 数组中。

In [20]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


**NumPy数组转张量**

In [21]:
n = np.ones(5)
t = torch.from_numpy(n)

In [24]:
np.add(n, 1, out=n)#numpy加法，数组n每个元素+1，原地操作，计算结果直接写回n本身，不新建数组
print(f"t: {t}")
print(f"n: {n}")

t: tensor([4., 4., 4., 4., 4.], dtype=torch.float64)
n: [4. 4. 4. 4. 4.]


**Learn the Basics** \|\| [Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Learn the Basics
================

Authors: [Suraj Subramanian](https://github.com/subramen), [Seth
Juarez](https://github.com/sethjuarez/), [Cassie
Breviu](https://github.com/cassiebreviu/), [Dmitry
Soshnikov](https://soshnikov.com/), [Ari
Bornstein](https://github.com/aribornstein/)

Most machine learning workflows involve working with data, creating
models, optimizing model parameters, and saving the trained models. This
tutorial introduces you to a complete ML workflow implemented in
PyTorch, with links to learn more about each of these concepts.

We\'ll use the FashionMNIST dataset to train a neural network that
predicts if an input image belongs to one of the following classes:
T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker,
Bag, or Ankle boot.

[This tutorial assumes a basic familiarity with Python and Deep Learning
concepts.]{.title-ref}

Running the Tutorial Code
-------------------------

You can run this tutorial in a couple of ways:

-   **In the cloud**: This is the easiest way to get started! Each
    section has a \"Run in Google Colab\" link at the top, which opens
    an integrated notebook in Google Colab with the code in a
    fully-hosted environment.
-   **Locally**: This option requires you to set up PyTorch and
    TorchVision first on your local machine ([installation
    instructions](https://pytorch.org/get-started/locally/)). Download
    the notebook or copy the code into your favorite IDE.

How to Use this Guide
---------------------

If you\'re familiar with other deep learning frameworks, check out the
[0. Quickstart](quickstart_tutorial.html) first to quickly familiarize
yourself with PyTorch\'s API.

If you\'re new to deep learning frameworks, head right into the first
section of our step-by-step guide: [1. Tensors](tensorqs_tutorial.html).

::: {.toctree maxdepth="2" hidden=""}
quickstart\_tutorial tensorqs\_tutorial data\_tutorial
transforms\_tutorial buildmodel\_tutorial autogradqs\_tutorial
optimization\_tutorial saveloadrun\_tutorial
:::
